# Text Classification with NLLB-200 Embeddings

Sentence embeddings can be used as fixed-size feature vectors for any
downstream classifier. The approach has several practical advantages:

1. **No labelled data at inference time** — embeddings are obtained from a
   pre-trained model; only the classifier head needs labelled training examples.
2. **Small training sets suffice** — 20–100 labelled examples often produce
   reasonable results, because the embedding space already encodes semantics.
3. **Domain transferability** — a classifier trained on one domain (e.g., product
   reviews) can partially generalise to related domains.

This notebook demonstrates two binary classification tasks:

- **Sentiment analysis** (positive / negative) using Turkish movie and product reviews.
- **Spam / ham detection** using Turkish SMS and email text samples.

Both tasks follow the same three-step recipe:
  1. Embed labelled sentences with the `embeddings` processor.
  2. Train a logistic regression classifier on the embeddings.
  3. Evaluate and run inference on new sentences.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import math, random
import turkicnlp
from turkicnlp import Pipeline

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report
    from sklearn.model_selection import train_test_split
except ImportError:
    raise SystemExit("Install scikit-learn: pip install scikit-learn")

turkicnlp.download("tur", processors=["embeddings"])
embed = Pipeline("tur", processors=["embeddings"])

def get_embedding(text):
    return embed(text).embedding

## Part A — Sentiment Analysis

### A.1 Dataset

A small inline Turkish sentiment dataset (30 sentences per class).
Positive sentences (label 1) express satisfaction, enjoyment, or approval;
negative sentences (label 0) express disappointment, frustration, or disapproval.

For production use, replace this with a full corpus such as
[SentiTurca](https://github.com/mhbasaran/SentiTurca) (350 K Turkish tweets).

In [ ]:
SENTIMENT_DATA = [
    # Positive (1)
    ("Bu film gerçekten muhteşemdi, kesinlikle tavsiye ederim.", 1),
    ("Yemek çok lezzetliydi, restoran mükemmel.", 1),
    ("Ürün beklentilerimi tamamen karşıladı, çok memnunum.", 1),
    ("Harika bir tatildi, her şey mükemmeldi.", 1),
    ("Servis çok hızlı ve personel güler yüzlüydü.", 1),
    ("Bu kitabı okumak çok keyifliydi, tavsiye ederim.", 1),
    ("Müşteri hizmetleri sorunumu hızla çözdü.", 1),
    ("Konser fantastikti, sanatçı sahneyi terk etmek istemedi.", 1),
    ("Çocuklarım bu oyuncağı çok sevdi, tekrar alacağız.", 1),
    ("Otel çok temiz ve konforluydu.", 1),
    ("Kargo çok hızlı geldi, ürün kusursuzdu.", 1),
    ("Bu deneyim hayatımın en güzel anlarından biri oldu.", 1),
    ("Fiyatına göre kalitesi oldukça iyi.", 1),
    ("Arkadaşlarıma kesinlikle öneririm.", 1),
    ("Sipariş tam açıklandığı gibi geldi, mükemmel paketleme.", 1),
    ("Uygulama son derece kullanışlı ve hızlı.", 1),
    ("Ekip çok profesyoneldi, her konuda yardımcı oldular.", 1),
    ("Ürün kalitesi fiyatı ile tam orantılı, harika.", 1),
    ("Kafeye bayıldım, atmosfer çok sıcak.", 1),
    ("Kurs içeriği çok zengin ve öğretici.", 1),
    # Negative (0)
    ("Bu film tamamen zaman kaybıydı, berbat senaryo.", 0),
    ("Yemek soğuk geldi, tadı hiç iyi değildi.", 0),
    ("Ürün resimlerdeki gibi değildi, hayal kırıklığı.", 0),
    ("Tatil mahvoldu, otel çok kötüydü.", 0),
    ("Servis saatler sürdü, kimse ilgilenmedi.", 0),
    ("Kitap çok sıkıcıydı, yarısını okuyamadım.", 0),
    ("Müşteri hizmetleri hiç yardımcı olmadı.", 0),
    ("Konser iptal edildi, para iadesi yapılmadı.", 0),
    ("Oyuncak çok hızlı bozuldu, kalitesiz.", 0),
    ("Oda çok pistı ve kokmaya başlamıştı.", 0),
    ("Kargo 3 hafta sonra geldi, ürün hasarlıydı.", 0),
    ("Bu deneyim için para harcadığıma pişmanım.", 0),
    ("Fiyatına göre kalitesi çok düşük.", 0),
    ("Kesinlikle tavsiye etmem, berbat bir hizmet.", 0),
    ("Sipariş kayboldu, kimse ilgilenmedi.", 0),
    ("Uygulama sürekli çöküyor, berbat.", 0),
    ("Ekip kaba ve umursamaz davrandı.", 0),
    ("Ürün sahte çıktı, dolandırıldım.", 0),
    ("Kafenin hijyeni çok kötüydü, yemek yiyemedim.", 0),
    ("Kurs vaat edilen içeriği sunmadı, pişmanım.", 0),
]

texts  = [t for t, _ in SENTIMENT_DATA]
labels = [l for _, l in SENTIMENT_DATA]
print(f"Dataset: {sum(labels)} positive, {len(labels)-sum(labels)} negative")

### A.2 Embed and Split

In [ ]:
print("Embedding training sentences... (may take ~1 min)")
X = [get_embedding(t) for t in texts]
y = labels

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

### A.3 Train and Evaluate

In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred,
                             target_names=["Negative", "Positive"]))

### A.4 Inference on New Sentences

In [ ]:
new_sentences = [
    "Bu ürünü çok beğendim, tekrar satın alacağım.",
    "Hiç memnun kalmadım, para israfı.",
    "Fena değildi ama daha iyi olabilirdi.",
    "Bu kadar kötü bir hizmet beklemiyordum.",
]

for sent in new_sentences:
    emb   = get_embedding(sent)
    prob  = clf.predict_proba([emb])[0]
    label = "Positive" if prob[1] >= 0.5 else "Negative"
    print(f"[{label} | P(pos)={prob[1]:.2f}]  {sent}")

## Part B — Spam / Ham Detection

### B.1 Dataset

A small inline dataset of Turkish SMS and email texts. Ham (legitimate, label 0)
includes meeting reminders, casual messages, and work requests. Spam (label 1)
includes unsolicited prize notifications, phishing attempts, and promotional offers.

For a larger benchmark consider the Turkish spam dataset on Kaggle or
use machine translation to project an English SMS spam corpus (e.g., UCI SMS Spam).

In [ ]:
SPAM_DATA = [
    # Ham (0)
    ("Yarınki toplantı 10:00'da, hazır olursun değil mi?", 0),
    ("Annem seni de akşam yemeğine davet ediyor.", 0),
    ("Projenin son halini gönderebilir misin?", 0),
    ("Bugün hava çok güzel, parka gidelim mi?", 0),
    ("Raporu dün bitirdim, review edebilir misin?", 0),
    ("Doktor randevum Perşembe saat 14:00.", 0),
    ("Mağazada indirim var, gidelim mi?", 0),
    ("Ödevi bitirdim, notlarını benimle paylaşır mısın?", 0),
    ("Yeni ev güzel, taşınma için yardım eder misin?", 0),
    ("Akşam film izlemeye gidiyoruz, gelir misin?", 0),
    ("Toplantı notlarını herkes ile paylaşabilir misin?", 0),
    ("Arabanı bu hafta kullanabilir miyim?", 0),
    ("Marketten süt ve ekmek alır mısın?", 0),
    ("Sınav notun belli oldu mu?", 0),
    ("Çocukları yarın okula sen mi götürüyorsun?", 0),
    # Spam (1)
    ("TEBRİKLER! 10.000 TL nakit ödülü kazandınız, hemen tıklayın!", 1),
    ("ÜCRETSİZ iPhone 15 kazanmak için linke tıklayın!", 1),
    ("Bankanızdan acil mesaj: Hesabınız askıya alınıyor!", 1),
    ("500 TL bonus kredi kartınıza yüklendi, aktifleştirin!", 1),
    ("ÖZEL TEKLİF: Bugün üye olun, yüzde seksen indirim kazanın!", 1),
    ("Çekiliş sonucu: Siz kazandınız! Ödülünüzü almak için tıklayın.", 1),
    ("Kripto ile bir haftada büyük kazanç elde edin, kaçırmayın!", 1),
    ("Hesabınızda şüpheli işlem tespit edildi, linke tıklayın!", 1),
    ("VIP üyelik bedava! Sadece bugün, hemen kaydolun!", 1),
    ("SEÇİLDİNİZ: 1000 TL hediye çeki sizi bekliyor!", 1),
    ("Banka bilgilerinizi güncelleyin, aksi hâlde hesabınız silinecek.", 1),
    ("Özel teklifimizden yararlanmak için son gün bugün!", 1),
    ("Tebrikler, piyango çekilişini kazandınız!", 1),
    ("Ücretsiz tatil paketi için bilgilerinizi girin.", 1),
    ("Anında kredi onayı, hiç belge istenmez!", 1),
]

spam_texts  = [t for t, _ in SPAM_DATA]
spam_labels = [l for _, l in SPAM_DATA]
print(f"Dataset: {spam_labels.count(0)} ham, {spam_labels.count(1)} spam")

### B.2 Train and Evaluate Spam Classifier

In [ ]:
print("Embedding spam/ham sentences...")
Xs = [get_embedding(t) for t in spam_texts]
ys = spam_labels

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    Xs, ys, test_size=0.25, random_state=42, stratify=ys)

clf_spam = LogisticRegression(max_iter=1000, random_state=42)
clf_spam.fit(Xs_train, ys_train)

ys_pred = clf_spam.predict(Xs_test)
print(classification_report(ys_test, ys_pred, target_names=["Ham", "Spam"]))

### B.3 Inference on New Messages

In [ ]:
new_msgs = [
    "Bu ay telefon faturanı ödemeyi unutma.",
    "KAZAN! Çekilişimizde büyük ödüller sizi bekliyor!",
    "Yarın spor salonunda görüşürüz.",
    "Hesabınız güvenlik nedeniyle kısıtlandı, acil işlem yapın!",
]

for msg in new_msgs:
    emb   = get_embedding(msg)
    prob  = clf_spam.predict_proba([emb])[0]
    label = "SPAM" if prob[1] >= 0.5 else "ham"
    print(f"[{label} | P(spam)={prob[1]:.2f}]  {msg}")